In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'data').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / 'data').exists():
    raise FileNotFoundError('Não foi possível localizar a raiz do projeto.')

for candidate in [
    PROJECT_ROOT / 'code',
    PROJECT_ROOT / 'code' / 'revenue',
    PROJECT_ROOT / 'code' / 'tmdb',
]:
    if candidate.exists() and str(candidate.resolve()) not in sys.path:
        sys.path.append(str(candidate.resolve()))

import requests
import pandas as pd
from tqdm import tqdm
import os
import time

# **Mineirando os Dados**

Este notebook realiza a etapa de coleta da base de dados utilizada no trabalho. O objetivo aqui é consultar a API pública do **The Movie Database (TMDB)** para obter um conjunto inicial de filmes e, para cada obra recuperada, reunir atributos descritivos e financeiros que serão usados posteriormente nas etapas de análise, pré-processamento e modelagem.


## **Descrição dos Atributos da Base de Dados**

Os atributos coletados foram escolhidos por serem metadados estruturados disponíveis na API e por terem potencial relevância para o problema de predição de receita. A tabela abaixo resume o significado de cada campo armazenado no conjunto bruto:

| Atributo | Descrição |
|----------|-----------|
| `id_tmdb` | Identificador único do filme na base de dados do TMDB. |
| `title` | Título original do filme, sem tradução. |
| `original_language` | Código do idioma original do filme, como `'en'` para inglês ou `'pt'` para português. |
| `adult` | Indicador booleano que informa se a obra está marcada como conteúdo adulto. |
| `video` | Indicador booleano que informa se a entrada representa um vídeo, como trailer ou clipe, e não propriamente um filme. |
| `genres` | Lista de gêneros associados ao filme, retornada como uma lista de dicionários contendo identificador e nome de cada gênero. |
| `status` | Situação de lançamento da obra na base, como `'Released'` ou `'Post Production'`. |
| `runtime` | Duração do filme em minutos. |
| `belongs_to_collection` | Informação sobre a franquia ou coleção da qual o filme faz parte, quando essa informação existir. |
| `budget` | Orçamento estimado de produção do filme, em dólares americanos. |
| `revenue` | Receita mundial total arrecadada pelo filme, também em dólares americanos. |


## **API**

A coleta é feita a partir da API v3 do TMDB. Primeiro, define-se a chave de autenticação e a URL base do serviço. Em seguida, são utilizadas duas consultas complementares:

* uma consulta às listas paginadas de filmes (`popular` e `top_rated`), usadas para obter os identificadores das obras;
* uma consulta de detalhes por filme (`/movie/{id}`), usada para complementar cada registro com informações que não aparecem integralmente na listagem inicial, como gêneros, status, duração, coleção, orçamento e receita.

Essa estratégia permite construir uma base mais rica do que a obtida apenas com os resultados resumidos das páginas de listagem.

In [ ]:
API_KEY = 'YOUR_KEY'
BASE_URL = 'https://api.themoviedb.org/3'

In [3]:
def get_movie_details(movie_id):
    url = f"{BASE_URL}/movie/{movie_id}?api_key={API_KEY}&language=en-US"
    response = requests.get(url)
    return response.json()

# Function to fetch movies
def get_movies(pages=500):
    movies = []
    types_movies = ['popular', 'top_rated']

    for type_movies in types_movies:
      print(type_movies)

      for page in tqdm(range(1, pages+1)):
          url = f"{BASE_URL}/movie/{type_movies}?api_key={API_KEY}&language=en-US&page={page}"
          response = requests.get(url)

          if response.status_code != 200:
              print(f"Error on page {page}: {response.status_code} - {response.text}")
              continue

          data = response.json()

          if 'results' not in data:
              print(f"'results' not found on page {page}: {data}")
              continue

          for movie in data['results']:
              details = get_movie_details(movie['id'])

              movies.append({
                  # Identification and language
                  'id_tmdb': movie['id'],
                  'title': movie['original_title'],
                  'original_language': movie['original_language'],

                  # Content and classification
                  'adult': movie['adult'],
                  'video': movie['video'],
                  'genres': details['genres'],

                  # Release information
                  'status': details['status'],

                  # Runtime
                  'runtime': details['runtime'],

                  # Franchise
                  'belongs_to_collection': details['belongs_to_collection'],

                  # Financial data
                  'budget': details['budget'],
                  'revenue': details['revenue'],
              })

          time.sleep(0.25)

      if type_movies != types_movies[-1]:
        time.sleep(120)

    return pd.DataFrame(movies)

## **Coleta dos Registros**

A função `get_movies` percorre páginas das listas `popular` e `top_rated`, que retornam até 20 filmes por página. Para cada filme encontrado, uma nova requisição é feita para recuperar os detalhes completos do registro. Ao final, todos os dados são reunidos em um `DataFrame` do Pandas.

Além disso, o código inclui pequenas pausas entre as requisições para reduzir a chance de problemas com limite de uso da API. Depois da coleta, o conjunto bruto é salvo em arquivo `.csv`, servindo como ponto de partida para as próximas etapas do trabalho.

In [4]:
df = get_movies(pages=500)
df

popular


100%|██████████| 500/500 [21:32<00:00,  2.58s/it]


top_rated


100%|██████████| 500/500 [15:56<00:00,  1.91s/it]


,id_tmdb,title,original_language,adult,video,genres,status,runtime,belongs_to_collection,budget,revenue
0,552524,Lilo & Stitch,en,False,False,"[{'id': 10751, 'name': 'Family'}, {'id': 35, '...",Released,108,None,100000000,610800000
1,950387,A Minecraft Movie,en,False,False,"[{'id': 10751, 'name': 'Family'}, {'id': 35, '...",Released,101,"{'id': 1461530, 'name': 'The Minecraft Movie C...",150000000,947000000
2,1257960,सिकंदर,hi,False,False,"[{'id': 28, 'name': 'Action'}, {'id': 18, 'nam...",Released,133,None,23500000,24727058
3,574475,Final Destination Bloodlines,en,False,False,"[{'id': 27, 'name': 'Horror'}, {'id': 9648, 'n...",Released,110,"{'id': 8864, 'name': 'Final Destination Collec...",50000000,229314062
4,1197306,A Working Man,en,False,False,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'nam...",Released,116,None,40000000,98652557
...,...,...,...,...,...,...,...,...,...,...,...
19995,456048,The Humanity Bureau,en,False,False,"[{'id': 878, 'name': 'Science Fiction'}, {'id'...",Released,94,None,6000000,17544173
19996,889,The Flintstones in Viva Rock Vegas,en,False,False,"[{'id': 35, 'name': 'Comedy'}, {'id': 10751, '...",Released,90,"{'id': 351684, 'name': 'The Flintstones Collec...",83000000,59468275
19997,412547,Killing Gunther,en,False,False,"[{'id': 28, 'name': 'Action'}, {'id': 35, 'nam...",Released,93,None,0,197616
19998,74997,The Human Centipede 2 (Full Sequence),en,False,False,"[{'id': 18, 'name': 'Drama'}, {'id': 27, 'name...",Released,91,"{'id': 96671, 'name': 'The Human Centipede Col...",0,170323


In [ ]:
data_dir = PROJECT_ROOT / "data"
data_dir.mkdir(parents=True, exist_ok=True)

df.to_csv(data_dir / "TMDB_movies_original.csv", index=False)